# e11_finetune — train

보고서 약라벨 비교 실험입니다. 이 실험의 전 단계 출력과 고정된 공개 라벨을 연결합니다. gold58 개발용 CV이며 최종 제출 모델에는 허용된 학습 자료만 사용합니다.


In [ ]:
EXPERIMENT = {'id': 'e11_finetune', 'suite': 'weak', 'backbone': 'swin_t', 'feature_dim': 768, 'slices': 5, 'crop': 224, 'resize': 232, 'interpolation': 'bicubic', 'aggregation': 'class_mil', 'triplet': True, 'protocols': True, 'weak_labels': True, 'finetune': True, 'ensemble': False, 'backbone_epochs': 1, 'epochs': 20, 'batch_size': 64, 'lr': 0.001, 'weight_decay': 0.01, 'expected_training_studies': 4350, 'labels_sha256': '6f704a7bdb2f894cc49445b19ba7c4378c3f548d3449e00361e10044bee40920', 'train_dependencies': ['e10_class_mil'], 'pipeline_sha256': 'cec8f464400c52aa93d9b8afc846c9dbe363b3d1afffb314ac15d962fc6abc4f'}


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pydicom==3.0.2', 'python-gdcm==3.2.6'])


In [ ]:
"""Shared source embedded into each standalone experiment notebook."""

from pathlib import Path

import hashlib

import json

import math

import numpy as np

import pandas as pd

import torch

from torch import nn

PLANES = ("Sagittal", "Coronal", "Axial")

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

SEEDS = (42, 2026, 3407)

def file_hash(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def validate_bundle(path, expected_config):
    path = Path(path).resolve()
    manifest = json.loads((path / "assets.json").read_text(encoding="utf-8"))
    if manifest["experiment"] != expected_config:
        raise ValueError("Wrong experiment bundle: attach this experiment's completed training version.")
    files = manifest["files"]
    wheels = {p.relative_to(path).as_posix() for p in (path / "wheels").glob("*.whl")}
    declared_wheels = {name for name in files if name.startswith("wheels/") and name.endswith(".whl")}
    if (not {"encoder.ts", "model.pt"}.issubset(files) or wheels != declared_wheels
            or len(wheels) != 2
            or not any(name.startswith("wheels/pydicom-3.0.2-") for name in wheels)
            or not any(name.startswith("wheels/python_gdcm-3.2.6-") for name in wheels)):
        raise ValueError("Bundle must include hashes for both models and exactly the two required wheels.")
    for name, digest in manifest["files"].items():
        target = (path / name).resolve()
        if not target.is_relative_to(path) or not target.is_file() or file_hash(target) != digest:
            raise ValueError(f"Invalid or corrupted bundle file: {name}")
    return manifest

def find_data(input_root=Path("/kaggle/input")):
    for path in (input_root / "rsna-knee-abnormality-detection",
                 input_root / "competitions/rsna-knee-abnormality-detection"):
        if (path / "test.csv").is_file():
            return path
    raise FileNotFoundError("Attach the RSNA Knee competition data.")

def find_bundle(input_root=Path("/kaggle/input")):
    paths = list(input_root.glob("*/run/bundle/assets.json"))
    paths += list(input_root.glob("notebooks/*/*/run/bundle/assets.json"))
    paths += list(input_root.glob("kernels/*/*/run/bundle/assets.json"))
    if len(paths) != 1:
        raise ValueError("Attach exactly one completed experiment training output.")
    return paths[0].parent

def make_transform(config):
    from torchvision.transforms import InterpolationMode
    from torchvision.transforms._presets import ImageClassification
    return ImageClassification(crop_size=config["crop"], resize_size=config["resize"],
                               interpolation=InterpolationMode(config["interpolation"]), antialias=True)

def ordered_slices(folder):
    import pydicom
    paths = sorted(folder.glob("*.dcm"))
    if not paths:
        raise FileNotFoundError(f"No DICOM slices in {folder}")
    headers = [pydicom.dcmread(p, stop_before_pixels=True,
               specific_tags=["ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber"]) for p in paths]
    if all(hasattr(h, "ImagePositionPatient") and hasattr(h, "ImageOrientationPatient") for h in headers):
        orientation = np.asarray(headers[0].ImageOrientationPatient, dtype=float)
        normal = np.cross(orientation[:3], orientation[3:])
        if np.linalg.norm(normal) <= .9:
            raise ValueError("Invalid DICOM orientation.")
        coordinates = np.asarray([np.dot(np.asarray(h.ImagePositionPatient, dtype=float), normal) for h in headers])
    elif all(hasattr(h, "InstanceNumber") for h in headers):
        coordinates = np.asarray([float(h.InstanceNumber) for h in headers])
    else:
        raise ValueError(f"No slice position or InstanceNumber: {folder}")
    if not np.isfinite(coordinates).all():
        raise ValueError("Non-finite slice positions.")
    order = np.argsort(coordinates, kind="stable")
    coordinates = coordinates[order]
    span = np.ptp(coordinates)
    positions = (coordinates - coordinates.min()) / span if span > 0 else np.full(len(paths), .5)
    return [paths[i] for i in order], positions.astype(np.float32)

def image_tensor(path, transform):
    import pydicom
    ds = pydicom.dcmread(path)
    pixels = ds.pixel_array.astype(np.float32)
    if pixels.ndim != 2 or not np.isfinite(pixels).all():
        raise ValueError(f"Invalid MRI slice: {path}")
    pixels = pixels * float(getattr(ds, "RescaleSlope", 1)) + float(getattr(ds, "RescaleIntercept", 0))
    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        pixels = -pixels
    low, high = np.percentile(pixels, [1, 99])
    if high <= low:
        low, high = float(pixels.min()), float(pixels.max())
    if high <= low:
        raise ValueError(f"Constant MRI slice: {path}")
    pixels = np.clip((pixels - low) / (high - low), 0, 1)
    return transform(torch.from_numpy(pixels).float().unsqueeze(0).repeat(3, 1, 1))

@torch.inference_mode()
def study_tokens(uid, series, image_root, encoder, transform, config, device):
    count, dim = config["slices"], config["feature_dim"]
    features = torch.zeros(3, count, dim)
    mask = torch.zeros(3, count, dtype=torch.bool)
    positions = torch.zeros(3, count)
    rows = series.loc[series.StudyInstanceUID == uid]
    for plane_index, plane in enumerate(PLANES):
        choices = rows.loc[rows.Anatomical_Plane == plane].sort_values(
            ["Fluid_Sensitive", "Fat_Suppression", "SeriesInstanceUID"], ascending=[False, False, True])
        if choices.empty:
            continue
        paths, coords = ordered_slices(image_root / uid / choices.iloc[0].SeriesInstanceUID)
        indices = np.unique(np.rint(np.linspace(.25, .75, count) * (len(paths) - 1)).astype(int))
        batch = torch.stack([image_tensor(paths[i], transform) for i in indices]).to(device)
        output = encoder(batch).cpu().float()
        if output.shape != (len(indices), dim) or not torch.isfinite(output).all():
            raise ValueError("Unexpected encoder features.")
        features[plane_index, :len(indices)] = output
        mask[plane_index, :len(indices)] = True
        positions[plane_index, :len(indices)] = torch.from_numpy(coords[indices])
    if not mask.any():
        raise ValueError(f"No supported MRI planes for study {uid}")
    return features, mask, positions

def pooled_features(tokens, mask):
    clean = torch.where(mask[..., None], tokens, 0.)
    return (clean.sum(2) / mask.sum(2).clamp_min(1)[..., None]).flatten(1)

class StudyHead(nn.Module):
    def __init__(self, config, mean, std):
        super().__init__()
        self.mode = config["aggregation"]
        self.position = config["position"]
        self.register_buffer("mean", mean.clone())
        self.register_buffer("std", std.clone())
        if self.mode == "mean":
            self.classifier = nn.Linear(3 * config["feature_dim"], 12)
        elif self.mode == "attention":
            self.project = nn.Linear(config["feature_dim"], 64)
            layer = nn.TransformerEncoderLayer(64, 4, 128, dropout=0., batch_first=True, norm_first=True)
            self.attention = nn.TransformerEncoder(layer, 1, enable_nested_tensor=False)
            self.classifier = nn.Linear(3 * 64, 12)
            # Initialize common layers identically before adding optional plane parameters.
            self.plane_embedding = nn.Parameter(torch.zeros(3, 64)) if config["plane_embedding"] else None
            self.register_buffer("frequencies", torch.exp(torch.arange(0, 64, 2) * (-math.log(10000.) / 64)))
        else:
            raise ValueError(f"Unknown aggregation: {self.mode}")

    def forward(self, tokens, mask, positions):
        if self.mode == "mean":
            return self.classifier((pooled_features(tokens, mask) - self.mean) / self.std)
        clean = torch.where(mask[..., None], tokens, 0.)
        x = self.project((clean - self.mean) / self.std)
        if self.position:
            angles = positions[..., None] * (2 * math.pi) * self.frequencies
            x = x + torch.stack((angles.sin(), angles.cos()), dim=-1).flatten(-2)
        if self.plane_embedding is not None:
            x = x + self.plane_embedding[None, :, None, :]
        b, planes, slices, dim = x.shape
        flat_mask = mask.reshape(b * planes, slices)
        safe_mask = flat_mask.clone()
        safe_mask[~safe_mask.any(1), 0] = True
        x = self.attention(x.reshape(b * planes, slices, dim), src_key_padding_mask=~safe_mask)
        x = torch.where(flat_mask[..., None], x, 0.)
        vectors = x.sum(1) / flat_mask.sum(1).clamp_min(1)[:, None]
        return self.classifier(vectors.reshape(b, planes * dim))

def make_folds(ids, y):
    from sklearn.model_selection import KFold
    if len(set(ids)) != len(ids) or len(y) != len(ids) or not np.isin(y, [0, 1]).all():
        raise ValueError("Expected unique IDs and complete binary labels.")
    order = np.argsort(ids)
    folds = np.full(len(ids), -1, dtype=int)
    for fold, (train, valid) in enumerate(KFold(3, shuffle=True, random_state=42).split(order)):
        for index in (order[train], order[valid]):
            count = y[index].sum(0)
            if not ((count > 0) & (count < len(index))).all():
                raise ValueError("Each fold/label needs positives and negatives.")
        folds[order[valid]] = fold
    return folds

def metric_scores(y, logits):
    from sklearn.metrics import roc_auc_score
    probabilities = logits.sigmoid().numpy()
    aucs = roc_auc_score(y, probabilities, average=None)
    return {"macro_auc": float(np.mean(aucs)), "per_label_auc": dict(zip(LABELS, aucs.tolist())),
            "bce": nn.functional.binary_cross_entropy_with_logits(logits, torch.from_numpy(y)).item()}

def save_submission(ids, probabilities, sample_columns, target):
    probabilities = np.asarray(probabilities)
    if (not ids or len(set(ids)) != len(ids) or list(sample_columns) != ["StudyInstanceUID", *LABELS]
            or probabilities.shape != (len(ids), 12) or not np.isfinite(probabilities).all()
            or not ((probabilities >= 0) & (probabilities <= 1)).all()):
        raise ValueError("Invalid submission IDs, label order, shape or probabilities.")
    frame = pd.DataFrame(probabilities, columns=LABELS)
    frame.insert(0, "StudyInstanceUID", ids)
    target = Path(target)
    temporary = target.with_suffix(".csv.part")
    frame.to_csv(temporary, index=False)
    temporary.replace(target)
    return frame


In [ ]:
"""Report-supervised MRI experiments; shared verbatim by train and offline submit."""

import re

def make_targets(train, labels, config):
    train = train.sort_values('StudyInstanceUID').reset_index(drop=True).copy()
    if (train.StudyInstanceUID.isna().any() or not train.StudyInstanceUID.is_unique
            or labels.StudyInstanceUID.isna().any() or not labels.StudyInstanceUID.is_unique
            or not set(labels.StudyInstanceUID) <= set(train.StudyInstanceUID)):
        raise ValueError('Invalid, duplicate or foreign study IDs in label source.')
    required = [x + suffix for x in LABELS for suffix in ('', '__conf', '__verdict')]
    if not set(required) <= set(labels.columns):
        raise ValueError('Missing report label columns.')
    values = labels[[*LABELS, *[x+'__conf' for x in LABELS]]].to_numpy(dtype=np.float32)
    if not np.isfinite(values).all() or not ((values >= 0) & (values <= 1)).all():
        raise ValueError('Weak scores/confidences must be finite values in [0,1].')
    if not labels[[x+'__verdict' for x in LABELS]].isin(['YES', 'NO', 'UNK']).all().all():
        raise ValueError('Unknown report verdict.')
    observed = train[LABELS].notna()
    if (observed.any(axis=1) != observed.all(axis=1)).any():
        raise ValueError('Unexpected partially gold-labelled study.')
    gold = observed.all(axis=1).to_numpy()
    gold_y = train.loc[gold, LABELS].to_numpy(dtype=np.float32)
    if not np.isin(gold_y, [0, 1]).all():
        raise ValueError('Gold targets must be binary.')
    source = labels.set_index('StudyInstanceUID').reindex(train.StudyInstanceUID)
    y = source[LABELS].fillna(.5).to_numpy(dtype=np.float32)
    conf = source[[x+'__conf' for x in LABELS]].fillna(0).to_numpy(dtype=np.float32)
    known = source[[x+'__verdict' for x in LABELS]].isin(['YES', 'NO']).to_numpy()
    w = ((.25 + .75 * conf) * known).astype(np.float32)
    y[gold], w[gold] = gold_y, 3.
    reports = train.Report.fillna('').map(lambda s: re.sub(r'\s+', ' ', str(s)).strip().casefold())
    if reports.eq('').any():
        raise ValueError('Missing report: cannot audit report-group leakage.')
    groups = reports.map(lambda s: hashlib.sha256(s.encode()).hexdigest()).to_numpy()
    if len(set(groups[gold])) != int(gold.sum()):
        raise ValueError('Duplicate gold reports require grouped gold CV.')
    overlap = ~gold & np.isin(groups, groups[gold])
    keep = gold | (config['weak_labels'] & ~overlap & (w.sum(1) > 0))
    audit = {'gold_studies': int(gold.sum()), 'weak_studies': int((keep & ~gold).sum()),
             'weak_gold_report_overlap': int(overlap.sum()),
             'weak_without_known_targets': int((~gold & (w.sum(1) == 0)).sum()),
             'missing_source_rows': int(source.ACL.isna().sum()),
             'known_weak_targets': {name: int(((w[:, i] > 0) & keep & ~gold).sum())
                                    for i, name in enumerate(LABELS)},
             'unknown_policy': 'zero loss weight; never assume a negative',
             'teacher_gold_independence': 'not established; development CV only'}
    return train.loc[keep].reset_index(drop=True), y[keep], w[keep], gold[keep], groups[keep], audit

def find_report_labels(config, input_root=Path('/kaggle/input')):
    paths = [input_root / 'rsna-knee-llm-labels/report_labels_v2.csv',
             input_root / 'datasets/pilkwang/rsna-knee-llm-labels/report_labels_v2.csv']
    found = [p for p in paths if p.is_file()]
    if len(found) != 1 or file_hash(found[0]) != config['labels_sha256']:
        raise ValueError('Attach the pinned Pilkwang report label dataset; CSV hash must match.')
    return found[0]

def find_training_run(experiment_id, input_root=Path('/kaggle/input')):
    slug = 'rsna-knee-' + experiment_id.replace('_', '-') + '-train'
    paths = [input_root / slug / 'run', input_root / 'notebooks/choiyoungwhan' / slug / 'run',
             input_root / 'kernels/choiyoungwhan' / slug / 'run']
    found = [p for p in paths if (p / 'bundle/assets.json').is_file()]
    if len(found) != 1:
        raise ValueError(f'Attach exactly the pinned training output of {experiment_id}.')
    meta = json.loads((found[0] / 'bundle/assets.json').read_text())
    if meta['experiment']['id'] != experiment_id:
        raise ValueError('Dependency experiment mismatch.')
    return found[0]

def view_signature(config):
    return {key: config[key] for key in ('slices', 'triplet', 'protocols', 'crop', 'resize',
                                        'interpolation', 'pipeline_sha256')}

def study_views(uid, series, image_root, transform, config):
    slots = [(plane, fs) for plane in PLANES for fs in (True, False)] if config['protocols'] else [
        (plane, None) for plane in PLANES]
    count, size = config['slices'], config['crop']
    images = torch.zeros(len(slots), count, 3, size, size)
    mask = torch.zeros(len(slots), count, dtype=torch.bool)
    positions = torch.zeros(len(slots), count)
    rows = series.loc[series.StudyInstanceUID == uid]
    for slot, (plane, fs) in enumerate(slots):
        choices = rows.loc[rows.Anatomical_Plane == plane]
        if fs is not None:
            choices = choices.loc[choices.Fat_Suppression == fs]
        choices = choices.sort_values(['Fluid_Sensitive', 'Fat_Suppression', 'SeriesInstanceUID'],
                                      ascending=[False, False, True])
        if choices.empty:
            continue
        paths, coords = ordered_slices(image_root / uid / choices.iloc[0].SeriesInstanceUID)
        centers = np.unique(np.rint(np.linspace(.25, .75, count) * (len(paths)-1)).astype(int))
        decoded = {}
        for j, center in enumerate(centers):
            if config['triplet']:
                indices = np.clip([center-1, center, center+1], 0, len(paths)-1)
                for index in indices:
                    if index not in decoded:
                        decoded[index] = image_tensor(paths[index], lambda x: x)[0]
                # Each channel remains inside this series; never bridge two protocol slots.
                images[slot, j] = transform(torch.stack([decoded[index] for index in indices]))
            else:
                images[slot, j] = image_tensor(paths[center], transform)
            mask[slot, j], positions[slot, j] = True, float(coords[center])
    if not mask.any():
        raise ValueError(f'No supported MRI input for {uid}')
    return images, mask, positions

@torch.inference_mode()
def weak_tokens(uid, series, image_root, encoder, transform, config, device):
    images, mask, positions = study_views(uid, series, image_root, transform, config)
    valid = images[mask]
    features = torch.cat([encoder(chunk.to(device)).cpu().float() for chunk in valid.split(16)])
    if features.shape != (int(mask.sum()), config['feature_dim']) or not torch.isfinite(features).all():
        raise ValueError('Invalid encoder output.')
    result = torch.zeros(*mask.shape, config['feature_dim'])
    result[mask] = features
    return result, mask, positions

class WeakHead(nn.Module):
    def __init__(self, config, mean, std):
        super().__init__()
        slots, dim = (6 if config['protocols'] else 3), config['feature_dim']
        self.mode = config['aggregation']
        self.register_buffer('mean', mean.clone())
        self.register_buffer('std', std.clone())
        if self.mode == 'mean':
            self.classifier = nn.Linear(slots * dim, 12)
        elif self.mode == 'class_mil':
            self.attention = nn.Sequential(nn.Linear(dim, 128), nn.Tanh(), nn.Linear(128, 12))
            self.class_weight = nn.Parameter(torch.empty(12, slots * dim))
            self.class_bias = nn.Parameter(torch.zeros(12))
            nn.init.normal_(self.class_weight, std=.01)
        else:
            raise ValueError('Unsupported weak experiment head.')

    def forward(self, tokens, mask):
        clean = torch.where(mask[..., None], tokens, self.mean)
        x = (clean - self.mean) / self.std
        if self.mode == 'mean':
            pooled = x.sum(2) / mask.sum(2).clamp_min(1)[..., None]
            return self.classifier(pooled.flatten(1))
        attention = self.attention(x).masked_fill(~mask[..., None], -1e4).softmax(2)
        attention = attention * mask[..., None]
        pooled = torch.einsum('bskc,bskd->bscd', attention, x)
        return (pooled.permute(0, 2, 1, 3).flatten(2) * self.class_weight).sum(-1) + self.class_bias

def weighted_bce(logits, targets, weights):
    loss = torch.nn.functional.binary_cross_entropy_with_logits(logits.float(), targets.float(), reduction='none')
    return (loss * weights).sum() / weights.sum().clamp_min(1e-8)

def fit_weak_head(tokens, mask, y, weights, train_index, seed, config):
    torch.manual_seed(seed)
    index = torch.as_tensor(train_index, dtype=torch.long, device=tokens.device)
    statistics = tokens[index][mask[index]]
    if not torch.isfinite(statistics).all() or len(statistics) < 2:
        raise ValueError('Invalid training features.')
    mean, std = statistics.mean(0), statistics.std(0).clamp_min(.01)
    del statistics
    head = WeakHead(config, mean, std).to(tokens.device)
    optimizer = torch.optim.AdamW(head.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    history = []
    for epoch in range(1, config['epochs'] + 1):
        order = index[torch.randperm(len(index), device=index.device)]
        total_loss = total_weight = 0.
        for batch in order.split(config['batch_size']):
            optimizer.zero_grad(set_to_none=True)
            loss = weighted_bce(head(tokens[batch], mask[batch]), y[batch], weights[batch])
            if not torch.isfinite(loss):
                raise ValueError('Non-finite training loss.')
            loss.backward()
            optimizer.step()
            batch_weight = weights[batch].sum().item()
            total_loss += loss.item() * batch_weight
            total_weight += batch_weight
        history.append({'epoch': epoch, 'train_loss': total_loss / total_weight,
                        'optimizer_steps': math.ceil(len(index) / config['batch_size'])})
    return head.eval(), history

def validate_weak_bundle(path, config):
    path = Path(path).resolve()
    manifest = json.loads((path / 'assets.json').read_text())
    if manifest['experiment'] != config or manifest['labels'] != LABELS:
        raise ValueError('Wrong weak experiment bundle.')
    required = ({f'members/{eid}/{name}' for eid in config['train_dependencies'] for name in ('encoder.ts','model.pt')}
                if config.get('ensemble') else {'encoder.ts', 'model.pt'})
    files = manifest['files']
    wheels = {p.relative_to(path).as_posix() for p in (path / 'wheels').glob('*.whl')}
    if (not required <= files.keys() or len(wheels) != 2 or not wheels <= files.keys()
            or not any(x.startswith('wheels/pydicom-3.0.2-') for x in wheels)
            or not any(x.startswith('wheels/python_gdcm-3.2.6-') for x in wheels)):
        raise ValueError('Missing weights or offline DICOM dependencies.')
    for name, digest in files.items():
        target = (path / name).resolve()
        if not target.is_relative_to(path) or not target.is_file() or file_hash(target) != digest:
            raise ValueError(f'Invalid bundle artifact: {name}')
    return manifest


In [ ]:
def build_encoder(pretrained=True):
    from torchvision.models import Swin_T_Weights, swin_t
    model = swin_t(weights=Swin_T_Weights.IMAGENET1K_V1 if pretrained else None)
    model.head = nn.Identity()
    return model


In [ ]:
"""Training entry point; embedded after the core and one backbone constructor."""

import importlib.metadata

import platform

import subprocess

import sys

import time

def export_encoder(encoder, config, target):
    torch.backends.mha.set_fastpath_enabled(False)
    encoder = encoder.cpu().eval().requires_grad_(False)
    size = config["crop"]
    example = torch.zeros(1, 3, size, size)
    with torch.no_grad():
        # Fixed H/W is part of the bundle contract; batch sizes are independently checked below.
        traced = torch.jit.trace(encoder, example, check_trace=False)
        for batch_size in range(1, config["slices"] + 1):
            probe = torch.rand(batch_size, 3, size, size)
            actual, expected = traced(probe), encoder(probe)
            print(f"Encoder export batch={batch_size}: max_abs_error={(actual - expected).abs().max().item():.8g}", flush=True)
            # Float32 eager/JIT execution can round differently near zero, as observed in DINOv2.
            torch.testing.assert_close(actual, expected, rtol=1e-4, atol=1e-4)
    traced.save(str(target))
    loaded = torch.jit.load(str(target), map_location="cpu").eval()
    with torch.no_grad():
        torch.testing.assert_close(loaded(probe), encoder(probe), rtol=1e-4, atol=1e-4)
    return loaded


In [ ]:
"""Train report-supervised ablations and save fully offline inference bundles."""

import shutil

import subprocess

import sys

import time

def write_weak_metrics(config, runs, out):
    metrics = {'experiment': config, 'runs': runs,
               'macro_auc_mean': float(np.mean([r['macro_auc'] for r in runs])),
               'macro_auc_seed_std': float(np.std([r['macro_auc'] for r in runs])),
               'fold_auc_mean': float(np.mean([f['macro_auc'] for r in runs for f in r['folds']])),
               'bce_mean': float(np.mean([r['bce'] for r in runs])),
               'note': 'Gold58 development OOF; public teacher independence is not established. Seed SD is not a CI.'}
    (out / 'metrics.json').write_text(json.dumps(metrics, indent=2))
    return metrics

def evaluate_weak_cv(tokens, mask, y, weights, gold, ids, config, out):
    gold_index = np.flatnonzero(gold)
    weak_index = np.flatnonzero(~gold)
    gold_ids = np.asarray(ids)[gold].tolist()
    gold_y = y[gold_index].cpu().numpy()
    folds = make_folds(gold_ids, gold_y)
    pd.DataFrame({'StudyInstanceUID': gold_ids, 'fold': folds}).to_csv(out / 'folds.csv', index=False)
    runs, predictions, histories = [], [], []
    for seed in SEEDS:
        oof = torch.full((len(gold_index), 12), float('nan'))
        scores = []
        for fold in range(3):
            train_index = np.concatenate([weak_index, gold_index[folds != fold]])
            valid = gold_index[folds == fold]
            head, history = fit_weak_head(tokens, mask, y, weights, train_index, seed, config)
            with torch.inference_mode():
                logits = head(tokens[valid], mask[valid]).cpu()
            oof[folds == fold] = logits
            scores.append({'fold': fold, **metric_scores(gold_y[folds == fold], logits)})
            histories.extend({'seed': seed, 'fold': fold, **row} for row in history)
            print(f"seed={seed} fold={fold} AUC={scores[-1]['macro_auc']:.5f}", flush=True)
            del head
        if not torch.isfinite(oof).all():
            raise ValueError('Incomplete gold OOF predictions.')
        runs.append({'seed': seed, **metric_scores(gold_y, oof), 'folds': scores})
        frame = pd.DataFrame(oof.sigmoid().numpy(), columns=LABELS)
        frame.insert(0, 'fold', folds)
        frame.insert(0, 'seed', seed)
        frame.insert(0, 'StudyInstanceUID', gold_ids)
        predictions.append(frame)
    pd.concat(predictions, ignore_index=True).to_csv(out / 'oof_predictions.csv', index=False)
    pd.DataFrame(histories).to_csv(out / 'history.csv', index=False)
    return write_weak_metrics(config, runs, out)

def freeze_swin_except_last_stage(encoder):
    encoder.eval().requires_grad_(False)
    encoder.features[-1].requires_grad_(True)
    encoder.norm.requires_grad_(True)
    return [p for p in encoder.parameters() if p.requires_grad]

def finetune_weak_encoder(encoder, cache, table, y, weights, gold, groups, series, data, config, out):
    device = 'cuda'
    weak = np.flatnonzero(~gold)
    valid = weak[np.array([int(group[:8], 16) % 10 == 0 for group in groups[weak]])]
    train_index = np.setdiff1d(weak, valid)
    if not len(valid) or not len(train_index) or gold[train_index].any():
        raise ValueError('Invalid weak-only encoder split.')
    torch.manual_seed(42)
    encoder = encoder.to(device)
    parameters = freeze_swin_except_last_stage(encoder)
    statistics = cache['tokens'][train_index][cache['mask'][train_index]]
    head = WeakHead(config, statistics.mean(0), statistics.std(0).clamp_min(.01)).to(device)
    del statistics
    optimizer = torch.optim.AdamW([{'params': parameters, 'lr': 1e-5},
                                  {'params': head.parameters(), 'lr': 1e-4}], weight_decay=.01)
    scaler = torch.amp.GradScaler('cuda')
    transform = make_transform(config)
    deadline = time.monotonic() + 7 * 3600
    best, best_epoch, history = float('inf'), None, []
    checkpoint = out / 'backbone_best.pt'

    def predict(index):
        if time.monotonic() > deadline:
            raise TimeoutError('Fine-tuning exceeded its 7h budget; no incomplete submission model will be used.')
        images, mask, _ = study_views(table.StudyInstanceUID.iloc[index], series, data / 'train_series', transform, config)
        mask = mask.to(device)
        with torch.autocast('cuda', dtype=torch.float16):
            valid_images = images[mask.cpu()].to(device)
            features = torch.cat([encoder(batch) for batch in valid_images.split(16)])
            tokens = features.new_zeros(*mask.shape, config['feature_dim'])
            tokens[mask] = features
            return head(tokens[None], mask[None])

    targets, confidence = torch.from_numpy(y).to(device), torch.from_numpy(weights).to(device)
    for epoch in range(1, config['backbone_epochs'] + 1):
        encoder.eval()
        encoder.features[-1].train()
        head.train()
        order = np.random.default_rng(42 + epoch).permutation(train_index)
        training_loss = 0.
        for start in range(0, len(order), 8):
            batch = order[start:start+8]
            optimizer.zero_grad(set_to_none=True)
            for index in batch:
                loss = weighted_bce(predict(index), targets[index:index+1], confidence[index:index+1])
                if not torch.isfinite(loss):
                    raise ValueError('Non-finite backbone training loss.')
                scaler.scale(loss / len(batch)).backward()
                training_loss += loss.item()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([*parameters, *head.parameters()], 5.)
            scaler.step(optimizer)
            scaler.update()
            if start % 400 == 0:
                print(f'Backbone epoch={epoch} studies={start}/{len(order)}', flush=True)
        encoder.eval()
        head.eval()
        validation_loss = validation_weight = 0.
        with torch.inference_mode():
            for index in valid:
                weight = confidence[index:index+1].sum().item()
                loss = weighted_bce(predict(index), targets[index:index+1], confidence[index:index+1]).item()
                validation_loss += loss * weight
                validation_weight += weight
        score = validation_loss / validation_weight
        if not np.isfinite(score):
            raise ValueError('Non-finite backbone validation loss.')
        history.append({'epoch': epoch, 'train_loss': training_loss / len(order), 'weak_validation_bce': score})
        if score < best:
            best, best_epoch = score, epoch
            torch.save({k: v.detach().cpu() for k, v in encoder.state_dict().items()}, checkpoint)
        print(f'Backbone epoch={epoch} weak validation BCE={score:.6f}', flush=True)
    encoder.load_state_dict(torch.load(checkpoint, map_location='cpu', weights_only=True))
    pd.DataFrame(history).to_csv(out / 'backbone_history.csv', index=False)
    report = {'training_weak_studies': len(train_index), 'validation_weak_studies': len(valid),
              'gold_studies_used_for_backbone': 0, 'best_epoch': best_epoch,
              'validation_bce': best, 'trainable_parameters': sum(p.numel() for p in parameters)}
    (out / 'backbone_audit.json').write_text(json.dumps(report, indent=2))
    del head, optimizer, scaler, targets, confidence
    return encoder.cpu().eval().requires_grad_(False)

def write_weak_manifest(config, bundle, table_count, gold_count, extra=None):
    manifest = {'experiment': config, 'labels': LABELS, 'python': list(sys.version_info[:2]),
                'torch': torch.__version__, 'final_training_studies': table_count,
                'gold_training_studies': gold_count, 'weak_training_studies': table_count-gold_count,
                'files': {p.relative_to(bundle).as_posix(): file_hash(p) for p in sorted(bundle.rglob('*'))
                          if p.is_file() and p != bundle / 'assets.json'}, **(extra or {})}
    (bundle / 'assets.json').write_text(json.dumps(manifest, indent=2))
    return manifest

def train_weak_ensemble(config, data, out):
    members = [find_training_run(eid) for eid in config['train_dependencies']]
    frames = [pd.read_csv(path / 'oof_predictions.csv').sort_values(['seed','StudyInstanceUID']).reset_index(drop=True)
              for path in members]
    pd.testing.assert_frame_equal(frames[0][['StudyInstanceUID','seed','fold']], frames[1][['StudyInstanceUID','seed','fold']])
    frame = frames[0].copy()
    frame[LABELS] = (frames[0][LABELS].to_numpy() + frames[1][LABELS].to_numpy()) / 2
    truth = pd.read_csv(data / 'train.csv').set_index('StudyInstanceUID')
    runs = []
    for seed, rows in frame.groupby('seed'):
        y = truth.loc[rows.StudyInstanceUID, LABELS].to_numpy(dtype=np.float32)
        logits = torch.logit(torch.from_numpy(rows[LABELS].to_numpy(dtype=np.float32)).clamp(1e-7, 1-1e-7))
        scores = [{'fold': fold, **metric_scores(y[rows.fold.to_numpy() == fold], logits[rows.fold.to_numpy() == fold])}
                  for fold in range(3)]
        runs.append({'seed': int(seed), **metric_scores(y, logits), 'folds': scores})
    frame.to_csv(out / 'oof_predictions.csv', index=False)
    shutil.copy2(members[0] / 'folds.csv', out / 'folds.csv')
    metrics = write_weak_metrics(config, runs, out)
    bundle = out / 'bundle'
    metadata = []
    for eid, path in zip(config['train_dependencies'], members):
        meta = json.loads((path / 'bundle/assets.json').read_text())
        validate_weak_bundle(path / 'bundle', meta['experiment'])
        metadata.append(meta)
        shutil.copytree(path / 'bundle', bundle / 'members' / eid)
    shutil.copytree(members[0] / 'bundle/wheels', bundle / 'wheels')
    if any(m['final_training_studies'] != config['expected_training_studies'] for m in metadata):
        raise ValueError('Ensemble members use different training populations.')
    write_weak_manifest(config, bundle, config['expected_training_studies'], 58,
                        {'members': config['train_dependencies'], 'mixing': 'fixed probability mean 0.5/0.5'})
    print(json.dumps(metrics, indent=2), flush=True)

def train_weak_experiment(config, encoder_factory, data, out):
    from tqdm.auto import tqdm
    data, out = Path(data), Path(out)
    if out.exists() and any(out.iterdir()):
        raise FileExistsError('Choose an empty output directory.')
    out.mkdir(parents=True, exist_ok=True)
    if config.get('ensemble'):
        return train_weak_ensemble(config, data, out)
    if not torch.cuda.is_available():
        raise RuntimeError('Run this notebook with a Kaggle GPU.')
    torch.set_num_threads(1)
    torch.manual_seed(42)
    torch.backends.mha.set_fastpath_enabled(False)
    label_path = find_report_labels(config)
    table, y, weights, gold, groups, audit = make_targets(pd.read_csv(data / 'train.csv'), pd.read_csv(label_path), config)
    if int(gold.sum()) != 58 or len(table) != config['expected_training_studies']:
        raise ValueError('Training population changed; review the label audit and pinned source.')
    audit.update({'source': 'pilkwang/rsna-knee-llm-labels/versions/1', 'sha256': file_hash(label_path)})
    (out / 'label_audit.json').write_text(json.dumps(audit, indent=2))
    print(json.dumps(audit), flush=True)
    series = pd.read_csv(data / 'train_series.csv')
    if not series.SeriesInstanceUID.is_unique or not set(table.StudyInstanceUID) <= set(series.StudyInstanceUID):
        raise ValueError('Missing or duplicate MRI series metadata.')
    bundle = out / 'bundle'
    bundle.mkdir()
    previous, cache = None, None
    if config['train_dependencies']:
        previous = find_training_run(config['train_dependencies'][0])
        previous_meta = json.loads((previous / 'bundle/assets.json').read_text())
        validate_weak_bundle(previous / 'bundle', previous_meta['experiment'])
        if file_hash(previous / 'tokens.pt') != previous_meta['tokens_sha256']:
            raise ValueError('Invalid cached feature artifact.')
        cache = torch.load(previous / 'tokens.pt', map_location='cpu', weights_only=True)
        shutil.copy2(previous / 'bundle/encoder.ts', bundle / 'encoder.ts')
        encoder = torch.jit.load(str(bundle / 'encoder.ts'), map_location='cpu').eval()
    else:
        encoder = export_encoder(encoder_factory(), {**config, 'slices': 16}, bundle / 'encoder.ts')
    if config['finetune']:
        if cache['ids'] != table.StudyInstanceUID.tolist() or cache['view_signature'] != view_signature(config):
            raise ValueError('Fine-tuning requires the matching preceding feature cache.')
        eager = encoder_factory(pretrained=False)
        eager.load_state_dict(encoder.state_dict(), strict=True)
        eager = finetune_weak_encoder(eager, cache, table, y, weights, gold, groups, series, data, config, out)
        encoder = export_encoder(eager, {**config, 'slices': 16}, bundle / 'encoder.ts')
        cache = None
        del eager
    encoder = encoder.to('cuda').eval()
    encoder_hash = file_hash(bundle / 'encoder.ts')
    cached_index = {}
    if cache is not None and cache['encoder_sha256'] == encoder_hash and cache['view_signature'] == view_signature(config):
        if len(cache['ids']) != len(set(cache['ids'])):
            raise ValueError('Duplicate cached study IDs.')
        cached_index = {uid: i for i, uid in enumerate(cache['ids'])}
    transform, rows = make_transform(config), []
    started = time.monotonic()
    torch.cuda.reset_peak_memory_stats()
    for uid in tqdm(table.StudyInstanceUID, desc=config['id'] + ' MRI features'):
        if uid in cached_index:
            index = cached_index[uid]
            rows.append(tuple(cache[key][index] for key in ('tokens', 'mask', 'positions')))
        else:
            rows.append(weak_tokens(uid, series, data / 'train_series', encoder, transform, config, 'cuda'))
    tokens, mask, positions = [torch.stack([row[i] for row in rows]).clone() for i in range(3)]
    if not torch.isfinite(tokens).all():
        raise ValueError('Non-finite feature cache.')
    timing = {'feature_seconds': time.monotonic()-started, 'images': int(mask.sum()),
              'cached_studies': len(set(table.StudyInstanceUID) & cached_index.keys()),
              'peak_cuda_allocated_mb': torch.cuda.max_memory_allocated()/1024**2,
              'gpu': torch.cuda.get_device_name(0)}
    torch.save({'ids': table.StudyInstanceUID.tolist(), 'tokens': tokens, 'mask': mask, 'positions': positions,
                'view_signature': view_signature(config), 'encoder_sha256': encoder_hash}, out / 'tokens.pt')
    del cache, rows, encoder
    torch.cuda.empty_cache()
    tokens, mask = tokens.to('cuda'), mask.to('cuda')
    targets, confidence = torch.from_numpy(y).to('cuda'), torch.from_numpy(weights).to('cuda')
    metrics = evaluate_weak_cv(tokens, mask, targets, confidence, gold, table.StudyInstanceUID.tolist(), config, out)
    head, history = fit_weak_head(tokens, mask, targets, confidence, np.arange(len(table)), 42, config)
    torch.save({'state': {k:v.detach().cpu() for k,v in head.state_dict().items()}, 'experiment': config,
                'labels': LABELS}, bundle / 'model.pt')
    pd.DataFrame(history).to_csv(out / 'final_fit_history.csv', index=False)
    wheels = bundle / 'wheels'
    if previous:
        shutil.copytree(previous / 'bundle/wheels', wheels)
    else:
        wheels.mkdir()
        subprocess.check_call([sys.executable, '-m', 'pip', 'download', '-q', '--only-binary=:all:', '--no-deps',
                               '--dest', str(wheels), 'pydicom==3.0.2', 'python-gdcm==3.2.6'])
    write_weak_manifest(config, bundle, len(table), int(gold.sum()),
                        {'head_parameters': sum(p.numel() for p in head.parameters()),
                         'tokens_sha256': file_hash(out / 'tokens.pt'),
                         'folds_sha256': file_hash(out / 'folds.csv'), 'label_audit': audit})
    (out / 'timings.json').write_text(json.dumps(timing, indent=2))
    print(json.dumps({'macro_auc': metrics['macro_auc_mean'], 'timing': timing}), flush=True)


In [ ]:
train_weak_experiment(EXPERIMENT, build_encoder, find_data(), Path('/kaggle/working/run'))
